# Image Preprocessing/Cleaning/Resizing

This notebook ensures all the images are of 800 x 600 size and also creates the individual cell images in subfolders for later access during feature creation

In [1]:
import os
from PIL import Image

In [3]:
def resize_images_with_crop(input_dir, output_dir, target_width=800, target_height=600):
    """
    Resize images to 800x600, cropping from top-left if aspect ratio doesn't match 4:3.
    Skip portrait images and return them in a separate list.
    
    Args:
        input_dir: Directory containing input images
        output_dir: Directory to save resized images
        target_width: Target width (default 800)
        target_height: Target height (default 600)
    
    Returns:
        tuple: (processed_count, skipped_portrait_images)
    """
    target_aspect_ratio = target_width / target_height  # 4:3
    skipped_portrait = []
    processed_count = 0
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    for filename in os.listdir(input_dir):
        filepath = os.path.join(input_dir, filename)
        
        # Skip if not a file
        if not os.path.isfile(filepath):
            continue
            
        try:
            with Image.open(filepath) as img:
                width, height = img.size
                aspect_ratio = width / height
                
                # Check if portrait (height > width)
                if height > width:
                    skipped_portrait.append(filename)
                    continue
                
                # If aspect ratio matches target, just resize
                if abs(aspect_ratio - target_aspect_ratio) < 0.01:
                    resized_img = img.resize((target_width, target_height), Image.LANCZOS)
                else:
                    # Crop from top-left to match 4:3 aspect ratio
                    if aspect_ratio > target_aspect_ratio:
                        # Image is wider - crop width
                        new_width = int(height * target_aspect_ratio)
                        crop_box = (0, 0, new_width, height)
                    else:
                        # Image is taller - crop height
                        new_height = int(width / target_aspect_ratio)
                        crop_box = (0, 0, width, new_height)
                    
                    cropped_img = img.crop(crop_box)
                    resized_img = cropped_img.resize((target_width, target_height), Image.LANCZOS)
                
                # Save the resized image
                output_path = os.path.join(output_dir, filename)
                resized_img.save(output_path)
                processed_count += 1
                
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    return processed_count, skipped_portrait

In [ ]:
RAW_IMAGES_PATH = 'data/final_data/raw_images/'
RESIZED_IMAGES_PATH = 'data/final_data/resized_images/'

if os.path.exists(RESIZED_IMAGES_PATH):
    print(f"Resized images directory '{RESIZED_IMAGES_PATH}' already exists. Skipping processing.")
else:
    processed, skipped = resize_images_with_crop(RAW_IMAGES_PATH, RESIZED_IMAGES_PATH)
    print(f"Processed: {processed} images")
    print(f"Skipped portrait images: {skipped}")

In [4]:
def divide_images_into_grid(input_dir, output_base_dir, grid_size=8):
    """
    Divide each image into an 8x8 grid (64 cells) and save sub-images in a folder hierarchy.
    
    Args:
        input_dir: Directory containing input images
        output_base_dir: Base directory for output sub-folders
        grid_size: Grid dimension (default 8 for 8x8 grid)
    
    Returns:
        dict: Summary with image names and number of cells created
    """
    processed_summary = {}
    
    for filename in os.listdir(input_dir):
        filepath = os.path.join(input_dir, filename)
        
        # Skip if not a file
        if not os.path.isfile(filepath):
            continue
            
        try:
            with Image.open(filepath) as img:
                width, height = img.size
                
                # Calculate cell dimensions
                cell_width = width // grid_size
                cell_height = height // grid_size
                
                # Create sub-folder for this image
                image_name = os.path.splitext(filename)[0]
                image_output_dir = os.path.join(output_base_dir, image_name)
                os.makedirs(image_output_dir, exist_ok=True)
                
                cell_count = 0
                
                # Divide into grid
                for row in range(grid_size):
                    for col in range(grid_size):
                        # Calculate cell number (1-based, row-major order)
                        cell_num = row * grid_size + col + 1
                        
                        # Calculate crop box
                        left = col * cell_width
                        top = row * cell_height
                        right = left + cell_width
                        bottom = top + cell_height
                        
                        # Crop and save cell
                        cell_img = img.crop((left, top, right, bottom))
                        cell_filename = f"c{cell_num:02d}.jpg"
                        cell_path = os.path.join(image_output_dir, cell_filename)
                        cell_img.save(cell_path)
                        
                        cell_count += 1
                
                processed_summary[filename] = cell_count
                print(f"Processed {filename}: {cell_count} cells created")
                
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    return processed_summary

In [ ]:
GRID_CELLS_PATH = 'data/final_data/grid_cells/'    
summary = divide_images_into_grid(RESIZED_IMAGES_PATH, GRID_CELLS_PATH)
print(f"\nTotal images processed: {len(summary)}")
print(f"Total cells created: {sum(summary.values())}")